In [1]:
# 加载环境变量
from dotenv import load_dotenv
from langchain_core.messages import AIMessage

load_dotenv()

True

# 定义模型
我们定义两个模型，一个多模态模型，一个聊天模型


In [8]:
from langchain.chat_models import init_chat_model
import os

# 多模态模型
multimodal_model = init_chat_model(
    model="qwen3.7-plus",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 定义工具


In [9]:
from langchain_tavily import TavilySearch

# web搜索工具，使用tavily作为web搜索工具
web_search = TavilySearch(
    max_results=5,
    topic="general"
)

# 添加记忆管理

这里我们采用sqllite

In [10]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 初始化checkpointer
checkpointer = SqliteSaver(sqlite3.connect("resources/personal_chief.db", check_same_thread=False))
# 自动建表
checkpointer.setup()

# 定义智能体

In [11]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，再搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    model=multimodal_model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

# 测试


In [16]:
from langchain.messages import HumanMessage
import base64
import httpx

image_url = "https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg"
image_response = httpx.get(image_url, follow_redirects=True)
image_response.raise_for_status()
image_b64 = base64.b64encode(image_response.content).decode("utf-8")

multimodal_message = HumanMessage(
    content=[
        {"type": "image_url",
         "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}},
        {"type": "text", "text": "帮我看看这些食材能做些什么？"}
    ])

config = {"configurable": {"thread_id": "4"}}

In [17]:
response = agent.invoke({"messages": [multimodal_message]}, config)

In [18]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================

## 🧊 食材识别结果

从照片中识别出以下新鲜食材：

| 类别 | 食材 | 新鲜度评估 |
|------|------|-----------|
| 🐟 蛋白质 | 三文鱼、鸡胸肉 | 色泽鲜亮，非常新鲜 |
| 🥦 蔬菜 | 西兰花、生菜 | 翠绿饱满，状态良好 |
| 🍄 菌菇 | 口蘑/蘑菇 | 饱满完整，品质好 |
| 🍅 果蔬 | 小番茄、红/黄甜椒、洋葱 | 色泽鲜艳，新鲜度高 |

---

现在让我基于这些食材搜索合适的食谱方案：
Tool Calls:
  tavily_search (call_0851eb74262e42c88f119e74)
 Call ID: call_0851eb74262e42c88f119e74
  Args:
    query: 三文鱼 西兰花 蘑菇 健康食谱 做法
    search_depth: basic
  tavily_search (call_e3415f1aac5e4c539b5d7c2e)
 Call ID: call_e3415f1aac5e4c539b5d7c2e
  Args:
    query: 鸡胸肉 甜椒 番茄 低脂高蛋白食谱
    search_depth: basic
  tavily_search (call_c7effe8272614ce890dbd541)
 Call ID: call_c7effe8272614ce890dbd541
  Args:
    query: 三文鱼 鸡胸肉 蔬菜沙拉 轻食食

In [19]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第3道菜，可以说的更详细点吗？")]},
    config
)

In [20]:
# 友好打印
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

# 🐟 烤三文鱼配蘑菇和烤西兰花 —— 详细版食谱

---

## 📋 食材清单（2人份）

| 食材 | 用量 | 备注 |
|------|------|------|
| 🐟 三文鱼排 | 2块（约300-400g） | 您的冰箱里有，厚度约2-3cm最佳 |
| 🥦 西兰花 | 1小颗（约200g） | 掰成小朵 |
| 🍄 蘑菇（口蘑） | 8-10个（约150g） | 切片或对半切 |
| 🫒 橄榄油 | 3汤匙 | 分蔬菜和三文鱼用 |
| 🧄 大蒜 | 3-4瓣 | 切碎或压泥 |
| 🍋 柠檬 | 半个 | 挤汁用 |
| 🧂 盐 | 适量 | 约1/2茶匙 |
| 🌶️ 黑胡椒 | 适量 | 现磨最佳 |
| 🌿 可选：迷迭香/百里香 | 少许 | 增加香气 |

---

## 🔧 工具准备

- ✅ 烤箱（或空气炸锅）
- ✅ 烤盘（建议铺锡纸或烘焙纸，方便清洗）
- ✅ 刷子（刷橄榄油用）
- ✅ 肉类温度计（可选，更精准）

---

## 👨‍🍳 详细步骤

### 第一步：预热烤箱（⏱️ 提前5分钟）

> 将烤箱预热至 **200°C（400°F）**
> 
> ⚠️ 一定要提前预热，否则食材受热不均匀

---

### 第二步：处理蔬菜（⏱️ 约5分钟）

1. **西兰花**：洗净后掰成大小均匀的小朵（约一口大小），太大会烤不透
2. **蘑菇**：用厨房纸擦干净表面（**不要水洗**，会吸水影响口感），对半切或切成厚片
3. **大蒜**：3-4瓣切碎备用

---

### 第三步：调味蔬菜（⏱️ 约2分钟）

将西兰花和蘑菇放入大碗中，加入：
- 🫒 橄榄油 **2汤匙**
- 🧄 蒜末 **2/3量**
-  盐 **1/4茶匙**
- 🌶️ 黑胡椒 **适量**
- 🌿 迷迭香/百里香（如有）

**充分翻拌均匀**，确保每块蔬菜都裹上油

---

### 第四步：调味三文鱼（⏱️ 约2分钟）

1. 用厨房纸**吸干三文鱼表面水分**（这一步很关键！水分多会影响焦香感）
2. 两面均匀涂抹：
   - 🫒 橄榄油 **1汤匙**
   - 🧂 盐 *

# 流式调用

有工具调用的agent消息类型比较多，要正确的流式输出比较麻烦，这里给出一个示例


In [21]:
from langchain.messages import AIMessageChunk

for chunk, metadata in agent.stream(
            {"messages": [multimodal_message]},
            config,
            stream_mode="messages"
        ):
            if isinstance(chunk, AIMessageChunk) and chunk.content:
                print(chunk.content, end="", flush=True)

# 🐟 烤三文鱼配蘑菇和烤西兰花 —— 详细版食谱

---

## 📋 食材清单（2人份）

| 食材 | 用量 | 备注 |
|------|------|------|
| 🐟 三文鱼排 | 2块（约300-400g） | 您的冰箱里有，厚度约2-3cm最佳 |
|  西兰花 | 1小颗（约200g） | 掰成小朵 |
| 🍄 蘑菇（口蘑） | 8-10个（约150g） | 切片或对半切 |
|  橄榄油 | 3汤匙 | 分蔬菜和三文鱼用 |
| 🧄 大蒜 | 3-4瓣 | 切碎或压泥 |
| 🍋 柠檬 | 半个 | 挤汁用 |
| 🧂 盐 | 适量 | 约1/2茶匙 |
| 🌶️ 黑胡椒 | 适量 | 现磨最佳 |
| 🌿 可选：迷迭香/百里香 | 少许 | 增加香气 |

---

## 🔧 工具准备

- ✅ 烤箱（或空气炸锅）
- ✅ 烤盘（建议铺锡纸或烘焙纸，方便清洗）
- ✅ 刷子（刷橄榄油用）
- ✅ 肉类温度计（可选，更精准）

---

## 👨‍ 详细步骤

### 第一步：预热烤箱（⏱️ 提前5分钟）

> 将烤箱预热至 **200°C（400°F）**
> 
> ⚠️ 一定要提前预热，否则食材受热不均匀

---

### 第二步：处理蔬菜（⏱️ 约5分钟）

1. **西兰花**：洗净后掰成大小均匀的小朵（约一口大小），太大会烤不透
2. **蘑菇**：用厨房纸擦干净表面（**不要水洗**，会吸水影响口感），对半切或切成厚片
3. **大蒜**：3-4瓣切碎备用

---

### 第三步：调味蔬菜（⏱️ 约2分钟）

将西兰花和蘑菇放入大碗中，加入：
- 🫒 橄榄油 **2汤匙**
-  蒜末 **2/3量**
- 🧂 盐 **1/4茶匙**
- 🌶️ 黑胡椒 **适量**
- 🌿 迷迭香/百里香（如有）

**充分翻拌均匀**，确保每块蔬菜都裹上油

---

### 第四步：调味三文鱼（⏱️ 约2分钟）

1. 用厨房纸**吸干三文鱼表面水分**（这一步很关键！水分多会影响焦香感）
2. 两面均匀涂抹：
   - 🫒 橄榄油 **1汤匙**
   - 🧂 盐 **1/4茶匙**
   - 🌶️ 黑胡椒 **适量**
   -  剩余蒜末
   - 🍋 挤少许柠檬汁

---

### 第五步：摆盘入烤箱（⏱️ 约1分钟）

在铺

# 获取会话历史

既然会话记忆交给了checkpointer管理，我们自然可以从中获取会话历史


In [22]:
checkpointer.get(config)

{'v': 4,
 'ts': '2026-06-17T01:02:11.092819+00:00',
 'id': '1f169e82-ba79-65b8-800b-d57779781db3',
 'channel_values': {'messages': [HumanMessage(content=[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}], additional_kwargs={}, response_metadata={}, id='b56e5497-6ec4-4269-bb4c-0a85a812e9f6'),
   AIMessage(content='## 🧊 食材识别结果\n\n从照片中识别出以下新鲜食材：\n\n| 类别 | 食材 | 新鲜度评估 |\n|------|------|-----------|\n| 🐟 蛋白质 | 三文鱼、鸡胸肉 | 色泽鲜亮，非常新鲜 |\n| 🥦 蔬菜 | 西兰花、生菜 | 翠绿饱满，状态良好 |\n| 🍄 菌菇 | 口蘑/蘑菇 | 饱满完整，品质好 |\n| 🍅 果蔬 | 小番茄、红/黄甜椒、洋葱 | 色泽鲜艳，新鲜度高 |\n\n---\n\n现在让我基于这些食材搜索合适的食谱方案：\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 475, 'prompt_tokens': 2254, 'total_tokens': 2729, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 174, 'rejected_prediction_tokens': None, 'text_tokens': 301}, 'p

In [23]:
for m in checkpointer.get(config)['channel_values']['messages']:
    print(type(m))
    print(m)

<class 'langchain_core.messages.human.HumanMessage'>
content=[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}] additional_kwargs={} response_metadata={} id='b56e5497-6ec4-4269-bb4c-0a85a812e9f6'
<class 'langchain_core.messages.ai.AIMessage'>
content='## 🧊 食材识别结果\n\n从照片中识别出以下新鲜食材：\n\n| 类别 | 食材 | 新鲜度评估 |\n|------|------|-----------|\n| 🐟 蛋白质 | 三文鱼、鸡胸肉 | 色泽鲜亮，非常新鲜 |\n| 🥦 蔬菜 | 西兰花、生菜 | 翠绿饱满，状态良好 |\n| 🍄 菌菇 | 口蘑/蘑菇 | 饱满完整，品质好 |\n| 🍅 果蔬 | 小番茄、红/黄甜椒、洋葱 | 色泽鲜艳，新鲜度高 |\n\n---\n\n现在让我基于这些食材搜索合适的食谱方案：\n\n' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 475, 'prompt_tokens': 2254, 'total_tokens': 2729, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 174, 'rejected_prediction_tokens': None, 'text_tokens': 301}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0,

定义一个便捷的获取会话历史的函数：


In [24]:
def get_messages(thread_id: str) -> list[dict[str, str]]:
    """获取会话历史"""

    # 根据 thread_id 查询 checkpoint
    cp = checkpointer.get({"configurable": {"thread_id": thread_id}})

    # 如果不存在，返回空列表
    if not cp:
        return []

    # 安全获取 messages
    channel_values = cp.get("channel_values")
    if not channel_values:
        return []

    messages = channel_values.get("messages", [])
    if not messages:
        return []

    # 转换消息格式
    result = []
    for msg in messages:
        if not msg.content:
            continue

        if isinstance(msg, HumanMessage):
            result.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            result.append({"role": "assistant", "content": msg.content})

    return result

In [25]:
get_messages("1")

[{'role': 'user',
  'content': [{'type': 'image',
    'url': 'https://tmp9527.oss-cn-beijing.aliyuncs.com/1772678117299.jpeg?x-oss-signature-version=OSS4-HMAC-SHA256&x-oss-date=20260305T082142Z&x-oss-expires=299&x-oss-credential=LTAI5tHzyC36KhUJfPqibaCo%2F20260305%2Fcn-beijing%2Foss%2Faliyun_v4_request&x-oss-signature=90083bdacfacc36386069d1e82c33b04ccd8eaa71c43fcf9248a36535d9cf46a'},
   {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]},
 {'role': 'user',
  'content': [{'type': 'image',
    'url': 'https://tmp9527.oss-cn-beijing.aliyuncs.com/1772678117299.jpeg?x-oss-signature-version=OSS4-HMAC-SHA256&x-oss-date=20260305T082142Z&x-oss-expires=299&x-oss-credential=LTAI5tHzyC36KhUJfPqibaCo%2F20260305%2Fcn-beijing%2Foss%2Faliyun_v4_request&x-oss-signature=90083bdacfacc36386069d1e82c33b04ccd8eaa71c43fcf9248a36535d9cf46a'},
   {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]},
 {'role': 'user',
  'content': [{'type': 'image',
    'url': 'https://tmp9527.oss-cn-beijing.aliyuncs.com/1772678117299.jpeg?x

# 清空会话历史


In [27]:
checkpointer.delete_thread("4")